In [1]:

import pandas as pd
import requests

url = "https://api.github.com/repos/diagram-chasing/cwc-flood-forecasts/contents/data"

response = requests.get(url)
response.raise_for_status()

files = pd.DataFrame(response.json())

print(files[['name', 'download_url']].to_string(index=False))


            name                                                                                     download_url
       README.md        https://raw.githubusercontent.com/diagram-chasing/cwc-flood-forecasts/main/data/README.md
    stations.csv     https://raw.githubusercontent.com/diagram-chasing/cwc-flood-forecasts/main/data/stations.csv
stations.geojson https://raw.githubusercontent.com/diagram-chasing/cwc-flood-forecasts/main/data/stations.geojson
stations.parquet https://raw.githubusercontent.com/diagram-chasing/cwc-flood-forecasts/main/data/stations.parquet


In [2]:

import pandas as pd

url = "https://raw.githubusercontent.com/diagram-chasing/cwc-flood-forecasts/main/data/stations.csv"

stations = pd.read_csv(url)

print("Dataset Shape:", stations.shape)
print("\nColumns:", stations.columns.tolist())

stations.head()


Dataset Shape: (1742, 11)

Columns: ['station_code', 'name', 'lat', 'lon', 'type', 'nearest_town', 'warning_level', 'danger_level', 'frl', 'mwl', 'highest_flow_level']


,station_code,name,lat,lon,type,nearest_town,warning_level,danger_level,frl,mwl,highest_flow_level
0,001-CDBNG,Shimoga,13.926944,75.585000,Base,NaN,566.7,567.25,NaN,NaN,570.18
1,001-CDJPR,Dhareri,23.147500,75.508333,Base,NaN,478.5,479.50,NaN,NaN,482.62
2,001-CHEJMU,Khanabal,33.763889,75.127500,Base,Khanabal,1591.2,1592.50,NaN,NaN,1595.43
3,001-DDASL,Tilaiya Reservoir,24.320000,85.520000,Inflow,NaN,NaN,NaN,372.475,371.465,372.28
4,001-ERDBWN,Balimundali,21.733889,86.633333,Base,NaN,NaN,NaN,NaN,NaN,43.10


In [3]:

import requests

url = "https://raw.githubusercontent.com/diagram-chasing/cwc-flood-forecasts/main/data/README.md"

response = requests.get(url)
print(response.text[:6000])


# Data

The dataset is three Parquet files, rebuilt monthly, whose columns are defined
in [DATA.md](../DATA.md).

| File | One row is | Size | View | Download |
| --- | --- | --- | --- | --- |
| `stations.parquet` | A station | 50 KB | [open](https://hyperparam.app/files?key=https://diagram-chasing.github.io/cwc-flood-forecasts/stations.parquet) | [stations.parquet](https://diagram-chasing.github.io/cwc-flood-forecasts/stations.parquet) |
| `daily.parquet` | A station, parameter and day, 1900 to today | 225 MB | [open](https://hyperparam.app/files?key=https://diagram-chasing.github.io/cwc-flood-forecasts/daily.parquet) | [daily.parquet](https://diagram-chasing.github.io/cwc-flood-forecasts/daily.parquet) |
| `hourly.parquet` | A reading, last three years | 94 MB | [open](https://hyperparam.app/files?key=https://diagram-chasing.github.io/cwc-flood-forecasts/hourly.parquet) | [hourly.parquet](https://diagram-chasing.github.io/cwc-flood-forecasts/hourly.parquet) |

The same files are atta

In [4]:

import duckdb

url = "https://diagram-chasing.github.io/cwc-flood-forecasts/daily.parquet"

data = duckdb.sql(f"""
    SELECT *
    FROM read_parquet('{url}')
    LIMIT 5
""").df()

print("Columns:", data.columns.tolist())
data.head()


Columns: ['station_code', 'datatype_code', 'variable', 'unit', 'source_type', 'date_ist', 'min', 'mean', 'max', 'sum', 'n_obs']


,station_code,datatype_code,variable,unit,source_type,date_ist,min,mean,max,sum,n_obs
0,001-CDBNG,BAT,bat,None,gauge,2021-03-03,10.0,10.000000,10.0,10.0,1
1,001-CDBNG,BAT,bat,None,gauge,2021-03-04,10.0,10.000000,10.0,10.0,1
2,001-CDBNG,BAT,bat,None,gauge,2021-03-05,10.1,13.166667,18.8,39.5,3
3,001-CDBNG,BAT,bat,None,gauge,2021-03-06,10.8,10.866667,11.0,32.6,3
4,001-CDBNG,BAT,bat,None,gauge,2021-03-07,10.1,12.800000,19.5,51.2,4


In [5]:

# Check available measurement types
variables = duckdb.sql(f"""
    SELECT
        datatype_code,
        variable,
        unit,
        COUNT(*) AS records
    FROM read_parquet('{url}')
    GROUP BY datatype_code, variable, unit
    ORDER BY records DESC
""").df()

variables


,datatype_code,variable,unit,records
0,HHS,water_level,m,8629192
1,MPS,rainfall,mm,7744877
2,MTX,air_temp_max,degC,3153727
3,MTN,air_temp_min,degC,3142205
4,HZS,water_level,m,3057780
...,...,...,...,...
76,HHY,hhy,None,49
77,MPH,mph,None,44
78,QWQ,qwq,None,34
79,MBS,mbs,None,33


In [6]:

# Extract rainfall and river water levels
flood_df = duckdb.sql(f"""
    SELECT
        station_code,
        date_ist,
        datatype_code,
        variable,
        unit,
        mean,
        max,
        sum,
        n_obs
    FROM read_parquet('{url}')
    WHERE datatype_code IN ('HHS', 'MPS')
""").df()

print("Dataset Shape:", flood_df.shape)
flood_df.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset Shape: (16374069, 9)


,station_code,date_ist,datatype_code,variable,unit,mean,max,sum,n_obs
0,001-CDBNG,1988-12-01,HHS,water_level,m,557.655000,557.66,1672.965,3
1,001-CDBNG,1988-12-02,HHS,water_level,m,557.688333,557.69,1673.065,3
2,001-CDBNG,1988-12-03,HHS,water_level,m,557.670000,557.67,1673.010,3
3,001-CDBNG,1988-12-04,HHS,water_level,m,557.690000,557.69,1673.070,3
4,001-CDBNG,1988-12-05,HHS,water_level,m,557.706667,557.71,1673.120,3


In [7]:

# Check coverage of each measurement
coverage = flood_df.groupby('variable').agg(
    Records=('station_code', 'size'),
    Stations=('station_code', 'nunique'),
    First_Date=('date_ist', 'min'),
    Last_Date=('date_ist', 'max')
)

print(coverage)

# Find stations with both rainfall and water-level data
station_types = flood_df.groupby('station_code')['variable'].nunique()

both_stations = station_types[station_types == 2].index

print("\nStations with both measurements:", len(both_stations))


             Records  Stations First_Date  Last_Date
variable                                            
rainfall     7744877      1551 1960-05-01 2026-10-27
water_level  8629192      1610 1919-06-16 2026-10-27

Stations with both measurements: 1535


In [8]:

# Separate water-level readings
water = flood_df[flood_df['variable'] == 'water_level'][
    ['station_code', 'date_ist', 'max']
].copy()

water = water.rename(columns={'max': 'water_level'})

# Separate rainfall readings
rain = flood_df[flood_df['variable'] == 'rainfall'][
    ['station_code', 'date_ist', 'sum']
].copy()

rain = rain.rename(columns={'sum': 'rainfall'})

# Combine using station and date
combined_df = pd.merge(
    water,
    rain,
    on=['station_code', 'date_ist'],
    how='inner'
)

print("Combined Dataset Shape:", combined_df.shape)
combined_df.head()


Combined Dataset Shape: (5482016, 4)


,station_code,date_ist,water_level,rainfall
0,001-CDBNG,1988-12-01,557.66,0.0
1,001-CDBNG,1988-12-02,557.69,0.0
2,001-CDBNG,1988-12-03,557.67,0.0
3,001-CDBNG,1988-12-04,557.69,0.0
4,001-CDBNG,1988-12-05,557.71,0.0


In [9]:

# Select station information
station_info = stations[[
    'station_code', 'name', 'lat', 'lon',
    'warning_level', 'danger_level'
]]

# Merge with historical data
flood_data = combined_df.merge(
    station_info,
    on='station_code',
    how='left'
)

print("Dataset Shape:", flood_data.shape)
print("\nMissing Values:")
print(flood_data.isnull().sum())

flood_data.head()


Dataset Shape: (5482016, 9)

Missing Values:
station_code           0
date_ist               0
water_level            0
rainfall               0
name               15440
lat                15440
lon                15440
warning_level    1491535
danger_level     1386981
dtype: int64


,station_code,date_ist,water_level,rainfall,name,lat,lon,warning_level,danger_level
0,001-CDBNG,1988-12-01,557.66,0.0,Shimoga,13.926944,75.585,566.7,567.25
1,001-CDBNG,1988-12-02,557.69,0.0,Shimoga,13.926944,75.585,566.7,567.25
2,001-CDBNG,1988-12-03,557.67,0.0,Shimoga,13.926944,75.585,566.7,567.25
3,001-CDBNG,1988-12-04,557.69,0.0,Shimoga,13.926944,75.585,566.7,567.25
4,001-CDBNG,1988-12-05,557.71,0.0,Shimoga,13.926944,75.585,566.7,567.25


In [10]:

# Remove records without required station information
flood_data = flood_data.dropna(subset=[
    'name', 'lat', 'lon',
    'warning_level', 'danger_level'
]).copy()

# Keep stations with valid threshold ordering
flood_data = flood_data[
    flood_data['warning_level'] < flood_data['danger_level']
].copy()

# Convert date column
flood_data['date_ist'] = pd.to_datetime(flood_data['date_ist'])

# Sort by station and date
flood_data = flood_data.sort_values(
    ['station_code', 'date_ist']
).reset_index(drop=True)

print("Cleaned Dataset Shape:", flood_data.shape)
print("Total Stations:", flood_data['station_code'].nunique())

flood_data.head()


Cleaned Dataset Shape: (3986675, 9)
Total Stations: 932


,station_code,date_ist,water_level,rainfall,name,lat,lon,warning_level,danger_level
0,001-CDBNG,1988-12-01,557.66,0.0,Shimoga,13.926944,75.585,566.7,567.25
1,001-CDBNG,1988-12-02,557.69,0.0,Shimoga,13.926944,75.585,566.7,567.25
2,001-CDBNG,1988-12-03,557.67,0.0,Shimoga,13.926944,75.585,566.7,567.25
3,001-CDBNG,1988-12-04,557.69,0.0,Shimoga,13.926944,75.585,566.7,567.25
4,001-CDBNG,1988-12-05,557.71,0.0,Shimoga,13.926944,75.585,566.7,567.25


In [11]:

import numpy as np

# Create flood risk categories
flood_data['Risk_Level'] = np.select(
    [
        flood_data['water_level'] >= flood_data['danger_level'],
        flood_data['water_level'] >= flood_data['warning_level']
    ],
    ['High', 'Medium'],
    default='Low'
)

print(flood_data['Risk_Level'].value_counts())

flood_data[[
    'name', 'date_ist', 'water_level',
    'warning_level', 'danger_level', 'Risk_Level'
]].head()


Risk_Level
Low       3759486
Medium     133460
High        93729
Name: count, dtype: int64


,name,date_ist,water_level,warning_level,danger_level,Risk_Level
0,Shimoga,1988-12-01,557.66,566.7,567.25,Low
1,Shimoga,1988-12-02,557.69,566.7,567.25,Low
2,Shimoga,1988-12-03,557.67,566.7,567.25,Low
3,Shimoga,1988-12-04,557.69,566.7,567.25,Low
4,Shimoga,1988-12-05,557.71,566.7,567.25,Low


In [12]:

# Previous day's measurements
flood_data['Previous_Rainfall'] = flood_data.groupby('station_code')['rainfall'].shift(1)

flood_data['Previous_Water_Level'] = flood_data.groupby('station_code')['water_level'].shift(1)

# Change in water level
flood_data['Water_Level_Change'] = (
    flood_data['water_level'] - flood_data['Previous_Water_Level']
)

# Rainfall over the last 3 observations
flood_data['Rainfall_3Day'] = (
    flood_data.groupby('station_code')['rainfall']
    .transform(lambda x: x.rolling(3).sum())
)

# Next day's risk category
flood_data['Next_Day_Risk'] = (
    flood_data.groupby('station_code')['Risk_Level'].shift(-1)
)

# Ensure the next observation is actually tomorrow
next_date = flood_data.groupby('station_code')['date_ist'].shift(-1)

flood_data.loc[
    (next_date - flood_data['date_ist']).dt.days != 1,
    'Next_Day_Risk'
] = None

print(flood_data[[
    'date_ist', 'rainfall', 'water_level',
    'Water_Level_Change', 'Rainfall_3Day',
    'Next_Day_Risk'
]].head(10))


    date_ist  rainfall  water_level  Water_Level_Change  Rainfall_3Day  \
0 1988-12-01       0.0      557.660                 NaN            NaN   
1 1988-12-02       0.0      557.690               0.030            NaN   
2 1988-12-03       0.0      557.670              -0.020            0.0   
3 1988-12-04       0.0      557.690               0.020            0.0   
4 1988-12-05       0.0      557.710               0.020            0.0   
5 1988-12-06       0.0      557.700              -0.010            0.0   
6 1988-12-07       0.0      557.690              -0.010            0.0   
7 1988-12-08       0.0      557.680              -0.010            0.0   
8 1988-12-09       0.0      557.665              -0.015            0.0   
9 1988-12-10       0.0      557.610              -0.055            0.0   

  Next_Day_Risk  
0           Low  
1           Low  
2           Low  
3           Low  
4           Low  
5           Low  
6           Low  
7           Low  
8           Low  
9    

In [13]:

# Check consecutive dates
previous_date = flood_data.groupby('station_code')['date_ist'].shift(1)
two_days_ago = flood_data.groupby('station_code')['date_ist'].shift(2)

valid_dates = (
    (flood_data['date_ist'] - previous_date).dt.days.eq(1) &
    (previous_date - two_days_ago).dt.days.eq(1)
)

# Keep complete records with consecutive history
ml_data = flood_data[
    valid_dates & flood_data['Next_Day_Risk'].notna()
].dropna(subset=[
    'Previous_Rainfall',
    'Previous_Water_Level',
    'Water_Level_Change',
    'Rainfall_3Day'
]).copy()

print("ML Dataset Shape:", ml_data.shape)
print("\nNext-Day Risk Distribution:")
print(ml_data['Next_Day_Risk'].value_counts())


ML Dataset Shape: (3857499, 15)

Next-Day Risk Distribution:
Next_Day_Risk
Low       3637080
Medium     129647
High        90772
Name: count, dtype: int64


In [14]:

# Distance from flood thresholds
ml_data['Warning_Distance'] = (
    ml_data['warning_level'] - ml_data['water_level']
)

ml_data['Danger_Distance'] = (
    ml_data['danger_level'] - ml_data['water_level']
)

# Select model features
features = [
    'water_level',
    'rainfall',
    'Previous_Rainfall',
    'Previous_Water_Level',
    'Water_Level_Change',
    'Rainfall_3Day',
    'Warning_Distance',
    'Danger_Distance'
]

X = ml_data[features]
y = ml_data['Next_Day_Risk']

print("Features:", X.shape)
print("Target:", y.shape)


Features: (3857499, 8)
Target: (3857499,)


In [15]:

# Split using chronological dates
split_date = ml_data['date_ist'].quantile(0.8)

train = ml_data['date_ist'] < split_date
test = ml_data['date_ist'] >= split_date

X_train = X.loc[train]
X_test = X.loc[test]

y_train = y.loc[train]
y_test = y.loc[test]

print("Training Records:", len(X_train))
print("Testing Records:", len(X_test))
print("Split Date:", split_date)

print("\nTraining Risk Distribution:")
print(y_train.value_counts())

print("\nTesting Risk Distribution:")
print(y_test.value_counts())


Training Records: 3085734
Testing Records: 771765
Split Date: 2023-12-03 00:00:00

Training Risk Distribution:
Next_Day_Risk
Low       2898421
Medium     108037
High        79276
Name: count, dtype: int64

Testing Risk Distribution:
Next_Day_Risk
Low       738659
Medium     21610
High       11496
Name: count, dtype: int64


In [16]:

from sklearn.ensemble import RandomForestClassifier

# Sample up to 500,000 training records
sample = X_train.sample(
    n=min(500000, len(X_train)),
    random_state=42
)

X_sample = sample
y_sample = y_train.loc[sample.index]

# Train Random Forest
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model.fit(X_sample, y_sample)

print("Model trained successfully!")


Model trained successfully!


In [17]:

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predict on testing data
y_pred = model.predict(X_test)

# Calculate accuracy
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))

# Precision, Recall and F1-score
print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    labels=['Low', 'Medium', 'High'],
    zero_division=0
))

# Confusion Matrix
print("\nConfusion Matrix:")
print(confusion_matrix(
    y_test, y_pred,
    labels=['Low', 'Medium', 'High']
))


Accuracy: 0.9791

Classification Report:
              precision    recall  f1-score   support

         Low       1.00      0.98      0.99    738659
      Medium       0.60      0.87      0.71     21610
        High       0.87      0.83      0.85     11496

    accuracy                           0.98    771765
   macro avg       0.82      0.89      0.85    771765
weighted avg       0.98      0.98      0.98    771765


Confusion Matrix:
[[727344  10994    321]
 [  1726  18803   1081]
 [   549   1431   9516]]


In [18]:

from sklearn.metrics import accuracy_score, classification_report

# Baseline: tomorrow's risk = today's risk
baseline_pred = ml_data.loc[test, 'Risk_Level']

print("Baseline Accuracy:",
      round(accuracy_score(y_test, baseline_pred), 4))

print("\nBaseline Classification Report:")
print(classification_report(
    y_test,
    baseline_pred,
    labels=['Low', 'Medium', 'High'],
    zero_division=0
))


Baseline Accuracy: 0.9859

Baseline Classification Report:
              precision    recall  f1-score   support

         Low       0.99      0.99      0.99    738659
      Medium       0.77      0.77      0.77     21610
        High       0.83      0.83      0.83     11496

    accuracy                           0.99    771765
   macro avg       0.87      0.87      0.87    771765
weighted avg       0.99      0.99      0.99    771765



In [19]:

import pandas as pd

# Compare today's and tomorrow's risk
transitions = pd.crosstab(
    ml_data.loc[test, 'Risk_Level'],
    y_test,
    rownames=['Today'],
    colnames=['Tomorrow']
)

print(transitions)

# Count risk changes
risk_changes = (
    ml_data.loc[test, 'Risk_Level'] != y_test
).sum()

print("\nRisk Changes:", risk_changes)
print("Risk Change Percentage:",
      round(risk_changes / len(y_test) * 100, 2))


Tomorrow  High     Low  Medium
Today                         
High      9536     433    1535
Low        672  734625    3354
Medium    1288    3601   16721

Risk Changes: 10883
Risk Change Percentage: 1.41


In [20]:

# Today's risk in the test dataset
today_risk = ml_data.loc[test, 'Risk_Level']

# Cases that become High Risk tomorrow
new_high = (today_risk != 'High') & (y_test == 'High')

# How many did Random Forest predict correctly?
detected = (y_pred[new_high] == 'High').sum()

total = new_high.sum()

print("New High Risk Cases:", total)
print("Correctly Predicted:", detected)
print("Missed:", total - detected)

print("Detection Rate:",
      round(detected / total * 100, 2), "%")


New High Risk Cases: 1960
Correctly Predicted: 202
Missed: 1758
Detection Rate: 10.31 %


In [21]:

# Select stations that are not High Risk today
escalation_data = ml_data[
    ml_data['Risk_Level'] != 'High'
].copy()

# Target: Will the station become High Risk tomorrow?
escalation_data['New_High_Risk'] = (
    escalation_data['Next_Day_Risk'] == 'High'
).astype(int)

print("Dataset Shape:", escalation_data.shape)

print("\nTarget Distribution:")
print(escalation_data['New_High_Risk'].value_counts())

print("\nPercentage of New High Risk:")
print(
    escalation_data['New_High_Risk'].mean() * 100
)


Dataset Shape: (3766754, 18)

Target Distribution:
New_High_Risk
0    3754632
1      12122
Name: count, dtype: int64

Percentage of New High Risk:
0.3218155472855408


In [22]:

from sklearn.ensemble import RandomForestClassifier

# Same chronological split
train_es = escalation_data[
    escalation_data['date_ist'] < split_date
]

test_es = escalation_data[
    escalation_data['date_ist'] >= split_date
]

# Keep all positive training cases
positive = train_es[train_es['New_High_Risk'] == 1]

# Sample negative cases for faster training
negative = train_es[
    train_es['New_High_Risk'] == 0
].sample(n=200000, random_state=42)

# Combine training records
train_sample = pd.concat([positive, negative])

X_es_train = train_sample[features]
y_es_train = train_sample['New_High_Risk']

X_es_test = test_es[features]
y_es_test = test_es['New_High_Risk']

# Train the escalation model
escalation_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

escalation_model.fit(X_es_train, y_es_train)

print("Escalation Model Trained!")
print("Training Records:", len(X_es_train))
print("Testing Records:", len(X_es_test))
print("Positive Test Cases:", y_es_test.sum())


Escalation Model Trained!
Training Records: 210162
Testing Records: 760261
Positive Test Cases: 1960


In [23]:

from sklearn.metrics import classification_report, confusion_matrix

# Predict new High Risk cases
es_pred = escalation_model.predict(X_es_test)

# Evaluate model
print("Classification Report:")
print(classification_report(
    y_es_test,
    es_pred,
    target_names=['No Escalation', 'New High Risk'],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_es_test, es_pred))

# Calculate detection rate
actual_high = y_es_test == 1

detected = (es_pred[actual_high] == 1).sum()
total = actual_high.sum()

print("\nNew High Risk Cases:", total)
print("Correctly Detected:", detected)
print("Missed:", total - detected)
print("Detection Rate:", round(detected / total * 100, 2), "%")


Classification Report:
               precision    recall  f1-score   support

No Escalation       1.00      0.99      1.00    758301
New High Risk       0.20      0.51      0.29      1960

     accuracy                           0.99    760261
    macro avg       0.60      0.75      0.64    760261
 weighted avg       1.00      0.99      0.99    760261


Confusion Matrix:
[[754239   4062]
 [   959   1001]]

New High Risk Cases: 1960
Correctly Detected: 1001
Missed: 959
Detection Rate: 51.07 %


In [24]:

from sklearn.metrics import precision_score, recall_score, f1_score

# Predicted scores for new High Risk
probabilities = escalation_model.predict_proba(X_es_test)[:, 1]

# Compare different alert thresholds
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    predictions = (probabilities >= threshold).astype(int)

    precision = precision_score(y_es_test, predictions, zero_division=0)
    recall = recall_score(y_es_test, predictions, zero_division=0)
    f1 = f1_score(y_es_test, predictions, zero_division=0)

    false_alerts = ((predictions == 1) & (y_es_test == 0)).sum()

    print(
        f"Threshold: {threshold:.1f} | "
        f"Precision: {precision:.3f} | "
        f"Recall: {recall:.3f} | "
        f"F1: {f1:.3f} | "
        f"False Alerts: {false_alerts}"
    )


Threshold: 0.3 | Precision: 0.114 | Recall: 0.644 | F1: 0.194 | False Alerts: 9790
Threshold: 0.4 | Precision: 0.153 | Recall: 0.582 | F1: 0.242 | False Alerts: 6330
Threshold: 0.5 | Precision: 0.198 | Recall: 0.511 | F1: 0.285 | False Alerts: 4062
Threshold: 0.6 | Precision: 0.249 | Recall: 0.432 | F1: 0.316 | False Alerts: 2548
Threshold: 0.7 | Precision: 0.320 | Recall: 0.348 | F1: 0.333 | False Alerts: 1451
Threshold: 0.8 | Precision: 0.431 | Recall: 0.240 | F1: 0.308 | False Alerts: 621


In [25]:

# Split historical escalation data into training and validation
validation_date = train_es['date_ist'].quantile(0.8)

print("Validation Start Date:", validation_date)

print("\nTraining Records:",
      (train_es['date_ist'] < validation_date).sum())

print("Validation Records:",
      (train_es['date_ist'] >= validation_date).sum())

print("\nValidation High Risk Cases:",
      train_es.loc[
          train_es['date_ist'] >= validation_date,
          'New_High_Risk'
      ].sum())



Validation Start Date: 2021-06-30 00:00:00

Training Records: 2404635
Validation Records: 601858

Validation High Risk Cases: 1634


In [26]:

# Separate training and validation data
train_final = train_es[
    train_es['date_ist'] < validation_date
]

val_data = train_es[
    train_es['date_ist'] >= validation_date
]

# Keep all positive training cases
positive = train_final[
    train_final['New_High_Risk'] == 1
]

# Sample negative cases
negative = train_final[
    train_final['New_High_Risk'] == 0
].sample(n=200000, random_state=42)

train_sample = pd.concat([positive, negative])

# Prepare features
X_train_final = train_sample[features]
y_train_final = train_sample['New_High_Risk']

X_val = val_data[features]
y_val = val_data['New_High_Risk']

# Train model
escalation_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

escalation_model.fit(X_train_final, y_train_final)

print("Model Retrained Successfully!")
print("Training Samples:", len(X_train_final))
print("Validation Samples:", len(X_val))
print("Positive Validation Cases:", y_val.sum())


Model Retrained Successfully!
Training Samples: 208528
Validation Samples: 601858
Positive Validation Cases: 1634


In [27]:

from sklearn.metrics import precision_score, recall_score, f1_score

# Predict validation scores
val_prob = escalation_model.predict_proba(X_val)[:, 1]

# Compare thresholds
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:

    pred = (val_prob >= threshold).astype(int)

    precision = precision_score(y_val, pred, zero_division=0)
    recall = recall_score(y_val, pred, zero_division=0)
    f1 = f1_score(y_val, pred, zero_division=0)

    false_alerts = ((pred == 1) & (y_val == 0)).sum()

    print(
        f"Threshold: {threshold:.1f} | "
        f"Precision: {precision:.3f} | "
        f"Recall: {recall:.3f} | "
        f"F1: {f1:.3f} | "
        f"False Alerts: {false_alerts}"
    )


Threshold: 0.3 | Precision: 0.117 | Recall: 0.544 | F1: 0.193 | False Alerts: 6684
Threshold: 0.4 | Precision: 0.153 | Recall: 0.480 | F1: 0.233 | False Alerts: 4332
Threshold: 0.5 | Precision: 0.194 | Recall: 0.412 | F1: 0.263 | False Alerts: 2802
Threshold: 0.6 | Precision: 0.249 | Recall: 0.343 | F1: 0.288 | False Alerts: 1691
Threshold: 0.7 | Precision: 0.326 | Recall: 0.257 | F1: 0.287 | False Alerts: 869
Threshold: 0.8 | Precision: 0.418 | Recall: 0.165 | F1: 0.236 | False Alerts: 375


In [28]:

import joblib
from google.colab import files

joblib.dump(model, "risk_model.joblib")
joblib.dump(escalation_model, "escalation_model.joblib")

files.download("risk_model.joblib")
files.download("escalation_model.joblib")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>